In [1]:
!pip install opensearch-py

In [2]:
import boto3, json, time
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import OpenSearchVectorSearch
from langchain.chains import RetrievalQA
from langchain_aws import ChatBedrock, ChatBedrockConverse

In [3]:
def get_data_from_s3(bucket_name, key):
    s3 = boto3.client(
        's3',
        region_name=region,
    )
    response = s3.get_object(Bucket=bucket_name, Key=key)
    data = response['Body'].read().decode('utf-8')

    return data

In [4]:
def generate_embedding(text):
    """
    Call an Amazon Bedrock embedding model to generate an embedding for the provided text.
    Replace the model_id and adjust the payload/response extraction based on your actual model.
    """
    bedrock_client = boto3.client('bedrock-runtime')
    model_id = "amazon.titan-embed-text-v1"  # Using Amazon Titan text embedding model
    payload = {"inputText": text}
    response = bedrock_client.invoke_model(
        modelId=model_id,
        contentType='application/json',
        body=json.dumps(payload)
    )
    result = json.loads(response['body'].read().decode('utf-8'))
    embedding = result.get('embedding')
    return embedding

In [5]:
session = boto3.session.Session()
region = session.region_name
credentials = session.get_credentials()
awsauth = AWSV4SignerAuth(credentials, region, service='aoss')
aoss_client = session.client('opensearchserverless')

suffix = "demo"
bucket_name = "bucket-test-cj"
vector_store_name = f"bedrock-sample-rag-{suffix}"
index_name = f"bedrock-sample-index-{suffix}"

In [6]:
# Retrieve collection details programmatically
collection_response = aoss_client.batch_get_collection(names=[vector_store_name])
if 'collectionDetails' in collection_response and collection_response['collectionDetails']:
    collection_id = collection_response['collectionDetails'][0]['id']
    host = f"{collection_id}.{region}.aoss.amazonaws.com"
    print("Using host:", host)
else:
    raise ValueError("Could not retrieve collection details.")

Using host: 3e7z7ay9fy9a8vlrk4j4.us-west-2.aoss.amazonaws.com


In [7]:
oss_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

## ONLY RUN THIS BLOCK IF YOU NEED TO DELETE THE WHOLE DATABASE!!!

In [ ]:
import json
import time
from opensearchpy.exceptions import NotFoundError

# Define a match_all query to retrieve all documents
search_query = {
    "query": {
        "match_all": {}
    },
    "size": 100  # Adjust page size as needed
}

scroll_duration = "2m"
response = oss_client.search(index=index_name, body=json.dumps(search_query), scroll=scroll_duration)

scroll_id = response.get("_scroll_id")
documents = response["hits"]["hits"]

while documents:
    for doc in documents:
        doc_id = doc["_id"]
        # The ignore parameter prevents raising a NotFoundError for HTTP 404 responses.
        delete_response = oss_client.delete(index=index_name, id=doc_id, ignore=[404])
        print(f"Deleted document {doc_id}: {delete_response.get('result', 'not_found')}")
    
    response = oss_client.scroll(scroll_id=scroll_id, scroll=scroll_duration)
    scroll_id = response.get("_scroll_id")
    documents = response["hits"]["hits"]

print("All documents processed for deletion.")


In [8]:
s3_data = get_data_from_s3("bucket-test-cj", "full_text.txt")
s3_data[0:200]

'BILL Reports Second Quarter Fiscal Year 2025 Financial Results\n• Q2 Core Revenue Increased 16% Year-Over-Year\n• Q2 Total Revenue Increased 14% Year-Over-Year\nSAN JOSE, Calif.--(BUSINESS WIRE) – Februa'

In [9]:
splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_text(s3_data)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 37


In [ ]:
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i}...")
    
    # Generate the embedding vector using the provided function.
    embedding_vector = generate_embedding(chunk)
    
    # Create the document with both the original text and its embedding.
    document = {
        "text": chunk,
        "vector": embedding_vector
    }
    
    # Index the document into AOSS; the service will auto-generate a unique document ID.
    response = oss_client.index(index=index_name, body=json.dumps(document))
    
    # Optionally, print the response to verify that the chunk was indexed successfully.
    print(f"Indexed chunk {i}: {response.get('result', 'No result returned')}")

In [10]:
keyword = "Subscription"

query = {
    "query": {
        "match": {
            "text": keyword
        }
    }
}

# Execute the search query against your index
search_response = oss_client.search(index=index_name, body=json.dumps(query))

# Process the search response to print the text and its embedding vector
for hit in search_response["hits"]["hits"]:
    source = hit["_source"]
    print("Document found:")
    print("Text:", source["text"])
    print("Embedding:", source["vector"])

Document found:
Text: to-day financial workflow. We are moving fast to address a vast market opportunity to transform the financial operations for millions 
of SMBs.”
“In Q2, we delivered strong financial results, expanded our non-GAAP operating margin, and continued our track record of 
execution across the company,” said John Rettig, BILL President and CFO. “We are executing on our strategic priorities and are 
confident that our strong business model will allow us to drive years of durable growth, an attractive long-term profitability profile, 
and sustained value generation for shareholders.”
Financial Highlights for the Second Quarter of Fiscal 2025:
• Total revenue was $362.6 million, an increase of 14% year-over-year.
• Core revenue, which consists of subscription and transaction fees, was $319.6 million, an increase of 16% year-over-
year. Subscription fees were $67.7 million, up 7% year-over-year. Transaction fees were $251.9 million, up 19% year-
over-year.
Embedding: [0.3398

In [ ]:
class EmbeddingAdapter:
    def embed_documents(self, texts):
        return [generate_embedding(text) for text in texts]
    
    def embed_query(self, text):
        return generate_embedding(text)


embedding_adapter = EmbeddingAdapter()

In [ ]:
vectorstore = OpenSearchVectorSearch(
    opensearch_url=f"https://{host}",
    index_name=index_name, 
    client=oss_client,
    embedding_function=embedding_adapter,
)

In [ ]:
llm = ChatBedrock(
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    region_name=region,
    temperature=0.0,
)

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

In [ ]:
query = "What were the subscription fees?"
qa_chain.run(query)

In [ ]:
from botocore.client import Config

In [ ]:
bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 0})
bedrock_client = boto3.client('bedrock-runtime', region_name = region)
bedrock_agent_client = boto3.client("bedrock-agent-runtime",
                              config=bedrock_config, region_name = region)

In [ ]:
def retrieve(query, kbId, numberOfResults=5):
    return bedrock_agent_client.retrieve(
        retrievalQuery= {
            'text': query
        },
        knowledgeBaseId=kbId,
        retrievalConfiguration= {
            'vectorSearchConfiguration': {
                'numberOfResults': numberOfResults,
                'overrideSearchType': "HYBRID", # optional
            }
        }
    )

In [ ]:
response = bedrock_agent_client.createKnowledgeBase(
    name="MyKnowledgeBase", 
    description="A short description",
    # Possibly more config for your knowledge base...
)
kb_id = response["knowledgeBaseId"]
print("Created Knowledge Base ID:", kb_id)

In [ ]:
query = "What were the subscription fees?"
response = retrieve(query, kb_id, 5)
retrievalResults = response['retrievalResults']